# ACF-MAAR: Autocorrelation-Guided Multi-Window and AutoML-based Adaptable Regression

## Leveraging Autocorrelation for Adaptive Multi-Window based Regression in Streaming Data

**Authors:** Shahad Palathingal and Ebin Deni Raj  
**Affiliation:** Department of Computer Science Engineering, IIIT Kottayam, Kerala, India

---

---
## 1. Environment Setup & Imports

Install required packages and import all dependencies.

> **Important:** After running **Cell 1a** (installs), go to **Runtime → Restart session**, then run **Cell 1b** (patches) and **Cell 1c** (imports) in order.

In [ ]:
# ===== Cell 1a: INSTALL PACKAGES =====
# Run this cell ONCE, then restart runtime (Runtime → Restart session)

!pip install pycaret==3.3.2 --no-deps --quiet
!pip install river==0.21.2 --no-deps --quiet
!pip install shap==0.46.0 --no-deps --quiet
!pip install scikit-plot --quiet
!pip install -q imbalanced-learn ipywidgets cloudpickle deprecation xxhash category_encoders databricks-sdk

print("✓ All packages installed.")
print("⚠ NOW: Go to Runtime → Restart session, then run Cell 1b.")

In [ ]:
# ===== Cell 1b: PATCH COMPATIBILITY ISSUES =====
# Run this AFTER restarting the runtime

import site, os, glob

# --- Patch 1: Fix PyCaret version check (removes broken raise statement) ---
pycaret_init = os.path.join(site.getsitepackages()[0], 'pycaret', '__init__.py')
with open(pycaret_init, 'w') as f:
    f.write('''import sys
from pycaret.utils._show_versions import show_versions
version_ = "3.3.2"
__version__ = version_
__all__ = ["show_versions", "__version__"]
''')
print("✓ Patch 1/4: PyCaret version check removed.")

# --- Patch 2: Fix PyCaret memory.py (bytes_limit incompatibility with joblib) ---
memory_path = os.path.join(site.getsitepackages()[0], 'pycaret', 'internal', 'memory.py')
with open(memory_path, 'r') as f:
    content = f.read()
if 'bytes_limit' in content:
    content = content.replace(
        'return FastMemory(tmpdir, verbose=0, bytes_limit=DEFAULT_BYTES_LIMIT)',
        'return FastMemory(tmpdir, verbose=0)'
    )
    with open(memory_path, 'w') as f:
        f.write(content)
    print("✓ Patch 2/4: PyCaret memory.py bytes_limit fix applied.")
else:
    print("✓ Patch 2/4: PyCaret memory.py already clean.")

# --- Patch 3: Fix sklearn._print_elapsed_time (removed in newer sklearn) ---
import sklearn.utils
import sys

def _print_elapsed_time(source, message=None):
    if message is not None:
        print(f"[{source}] {message}")

sklearn.utils._print_elapsed_time = _print_elapsed_time
sys.modules['sklearn.utils']._print_elapsed_time = _print_elapsed_time
print("✓ Patch 3/4: sklearn._print_elapsed_time patched.")

# --- Patch 4: Fix scikitplot scipy.interp import ---
scikitplot_dir = os.path.join(site.getsitepackages()[0], 'scikitplot')
fixed_files = []
for filepath in glob.glob(os.path.join(scikitplot_dir, '*.py')):
    with open(filepath, 'r') as f:
        content = f.read()
    if 'from scipy import interp' in content:
        content = content.replace('from scipy import interp', 'from numpy import interp')
        with open(filepath, 'w') as f:
            f.write(content)
        fixed_files.append(os.path.basename(filepath))
if fixed_files:
    print(f"✓ Patch 4/4: Fixed scikitplot files: {', '.join(fixed_files)}")
else:
    print("✓ Patch 4/4: scikitplot already clean.")

print()
print("✅ All patches applied. Now run Cell 1c (imports).")

In [ ]:
# ===== Cell 1c: IMPORTS =====
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['PYTHONWARNINGS'] = 'ignore'

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import time
import threading

from sklearn.model_selection import cross_val_score, RepeatedKFold
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

from river import drift
from river.linear_model import LinearRegression
from river.optim import SGD
from river.metrics import MSE
from river.stream import iter_pandas

import jinja2
from pycaret.regression import setup, compare_models, pull, predict_model, finalize_model

from river.drift import PageHinkley, ADWIN, KSWIN

from datetime import datetime

from statsmodels.tsa.stattools import acf
from scipy.stats import chi2

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print("✅ All imports successful.")

In [ ]:
# ===== Cell 1d: MOUNT GOOGLE DRIVE =====
from google.colab import drive
drive.mount('/content/drive')

---
## 2. Data Loading & Pre-processing

Load the Individual Household Electric Power Consumption dataset (UCI ML Repository).  
The dataset contains 2,075,259 minute-level readings from December 2006 to November 2010.

**Target variable:** `Global_active_power` (kW)

In [ ]:
# --- Configure dataset path ---
src_minute = "/content/drive/MyDrive/AFC_MAAR/MINUTE_power_consumption_NEW.csv"

# --- Load dataset ---
df_minute = pd.read_csv(
    src_minute,
    infer_datetime_format=True,
    parse_dates=['Date_Time']
)
df_minute.set_index('Date_Time', drop=False, inplace=True)

print(f"Dataset shape: {df_minute.shape}")
print(f"Date range: {df_minute.index.min()} to {df_minute.index.max()}")
df_minute.head()

In [ ]:
# --- Extract target variable ---
newdf = df_minute[['Global_active_power']]
print(f"Target series shape: {newdf.shape}")
print(f"\nBasic statistics:")
newdf.describe()

---
## 3. Feature Engineering (Supervised Framing)

Transform the univariate time series into a supervised learning problem using lagged features.  
For a window size of `n_in` lags, each sample contains `var1(t-n_in), ..., var1(t-1)` as features  
and `var1(t)` as the target.

**Reference:** Section 3.1 (Preliminaries and Notation)

In [ ]:
def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):
    """
    Convert a time series to a supervised learning dataset.

    Parameters
    ----------
    data : array-like
        Sequence of observations.
    n_in : int
        Number of lag observations as input (X). Default: 1.
    n_out : int
        Number of observations as output (y). Default: 1.
    dropnan : bool
        Whether to drop rows with NaN values. Default: True.

    Returns
    -------
    pd.DataFrame
        Supervised learning DataFrame with lagged features.
    """
    n_vars = 1 if type(data) is list else data.shape[1]
    dff = pd.DataFrame(data)
    cols, names = list(), list()

    # Input sequence (t-n_in, ..., t-1)
    for i in range(n_in, 0, -1):
        cols.append(dff.shift(-i))
        names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]

    # Forecast sequence (t, t+1, ..., t+n_out-1)
    for i in range(0, n_out):
        cols.append(dff.shift(-i))
        if i == 0:
            names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
        else:
            names += [('var%d(t+%d)' % (j+1, i)) for j in range(n_vars)]

    agg = pd.concat(cols, axis=1)
    agg.columns = names
    if dropnan:
        agg.dropna(inplace=True)
    return agg

In [ ]:
# --- Create supervised dataset with 90 lagged features ---
N_LAGS = 90
reframed = series_to_supervised(newdf.values, N_LAGS, 1)
print(f"Supervised dataset shape: {reframed.shape}")
print(f"Features: var1(t-90) to var1(t-1)")
print(f"Target: var1(t)")
reframed.head()

---
## 4. Data Preparation & Train-Test Split

Extract 1 day of data (1440 minute-level samples) and split into features/target.  
The split preserves temporal ordering (no shuffling).

**Reference:** Section 4.1 (Experimental Setup)

In [ ]:
# --- Extract 1 day of data (1440 samples) ---
df = reframed.iloc[0:1440]
print(f"1-day data shape: {df.shape}")

# --- Separate features and target ---
X = df.drop(columns=['var1(t)'])
y = df['var1(t)']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# --- Temporal train-test split (80/20) ---
train_size = int(0.8 * len(df))  # 1152 samples

X_train = X.iloc[:train_size]
y_train = y.iloc[:train_size]
X_test = X.iloc[train_size:]
y_test = y.iloc[train_size:]

print(f"Training set: X={X_train.shape}, y={y_train.shape}")
print(f"Test set:     X={X_test.shape}, y={y_test.shape}")

In [ ]:
# --- Prepare full train/test DataFrames for streaming pipeline ---
train = reframed.drop("var1(t-90)", axis=1)
test = reframed[["var1(t-90)"]]

print(f"Full train features shape: {train.shape}")
print(f"Full test target shape: {test.shape}")

---
## 5. Initial AutoML Model Selection (Phase 1)

Use PyCaret AutoML to evaluate candidate regression models and select the best performer.  
The candidate pool includes: Extra Trees, XGBoost, Random Forest, Gradient Boosting,  
Huber Regressor, Linear Regression, Ridge, Bayesian Ridge, LAR, and OMP.

**Reference:** Section 3.4 (AutoML Optimization for Streaming)

In [ ]:
# --- AutoML model selection function ---
def automodel(data):
    """
    Run AutoML model selection on the given data window.

    Parameters
    ----------
    data : pd.DataFrame
        DataFrame containing features and target column 'var1(t-90)'.

    Returns
    -------
    model
        Finalized best model selected by PyCaret.
    """
    s = setup(
        data=data,
        target="var1(t-90)",
        session_id=123,
        verbose=False
    )

    models_to_compare = [
        'et',       # Extra Trees
        'xgboost',  # XGBoost
        'rf',       # Random Forest
        'gbr',      # Gradient Boosting
        'huber',    # Huber Regressor
        'lr',       # Linear Regression
        'ridge',    # Ridge Regression
        'br',       # Bayesian Ridge
        'lar',      # Least Angle Regression
        'omp'       # Orthogonal Matching Pursuit
    ]

    best = compare_models(include=models_to_compare, sort='MAE', fold=5)
    best_model = finalize_model(best)

    return best_model

In [ ]:
# --- Initial AutoML model training on Day 1 ---
print("Running AutoML with correct understanding...")
print(f"Target: var1(t-90)")
print(f"Features: var1(t-89) to var1(t)")
print()

s = setup(data=df, target="var1(t-90)", session_id=123)

models_to_compare = ['et', 'xgboost', 'rf', 'gbr', 'huber', 'lr', 'ridge', 'br', 'lar', 'omp']
best = compare_models(include=models_to_compare, sort='MAE', fold=5)
final_best_model = finalize_model(best)

model_name = type(final_best_model[-1]).__name__
print(f"\nBest model selected: {model_name}")

# Get trained feature names
trained_features = list(final_best_model[-1].feature_names_in_)
print(f"Trained on {len(trained_features)} features: {trained_features[0]} ... {trained_features[-1]}")

In [ ]:
# --- Evaluate on unseen Day 2 ---
eval_df = reframed.iloc[1440:2880]
eval_X = eval_df.drop(columns=['var1(t-90)'])
eval_y = eval_df['var1(t-90)']

predictions = final_best_model.predict(eval_X)

mae = mean_absolute_error(eval_y, predictions)
mse = mean_squared_error(eval_y, predictions)
r2 = r2_score(eval_y, predictions)
mape = mean_absolute_percentage_error(eval_y, predictions)

print(f"Initial Model Performance (on unseen Day 2):")
print(f"  MAE:  {mae:.4f}")
print(f"  MSE:  {mse:.4f}")
print(f"  R²:   {r2:.4f}")
print(f"  MAPE: {mape:.4f}")

---
## 6. Drift Detection Setup

Initialize three conventional drift detectors for comparison:
- **Page-Hinkley:** Monitors cumulative deviations from running mean (Eq. 2)
- **ADWIN:** Adaptive windowing with statistical bound on sub-window means (Eq. 1)
- **KSWIN:** Kolmogorov-Smirnov test between two windows

**Reference:** Section 3.3 (ACF-Based Structural Drift Detection)

In [ ]:
# --- Initialize drift detectors ---
ph = PageHinkley(min_instances=60, threshold=5)
ad = ADWIN(delta=0.004)
ks = KSWIN(alpha=0.001, window_size=500, stat_size=150)

methods = {
    "Page-Hinkley": ph,
    "ADWIN": ad,
    "KSWIN": ks
}

print("Drift detectors initialized:")
for name, detector in methods.items():
    print(f"  - {name}: {type(detector).__name__}")

---
## 7. Drift Detection Evaluation

Run all three detectors on the target stream and count detected drifts.  
This corresponds to the drift detection quality comparison in Table 5.

**Reference:** Section 4.3 (Drift Detection Quality)

In [ ]:
# --- Run drift detectors on target stream ---
target_values = reframed['var1(t)'].iloc[0:1440].values.flatten()

drift_log = {name: [] for name in methods}

for idx, val in enumerate(target_values):
    for name, dd in methods.items():
        dd.update(val)
        if dd.drift_detected:
            drift_log[name].append(idx)

# --- Summary ---
total_drifts = sum(len(d) for d in drift_log.values())

print("Drift Detection Results (1-day window):")
print(f"{'Detector':<25} {'Drifts Detected':>15}")
print("-" * 42)
for name, drifts in drift_log.items():
    print(f"{name:<25} {len(drifts):>15}")
print("-" * 42)
print(f"{'Total':<25} {total_drifts:>15}")

---
## 8. ACF-MAAR Streaming Pipeline (Algorithm 1)

The main streaming loop implements the two-tier adaptation policy:
- **Tier 1:** If drift is detected but prediction error is within tolerance → continue with current model
- **Tier 2:** If drift is detected AND error exceeds threshold → trigger AutoML model replacement

This cell implements Algorithm 1 from Section 3.6.

**Reference:** Section 3.5 (Autocorrelation-Triggered Model Adaptation) & Section 3.6 (ACF-MAAR Algorithm)

In [ ]:
# --- ACF-MAAR Streaming Pipeline ---
DAY_SIZE = 1440
ERROR_THRESHOLD = 0.2
START_IDX = DAY_SIZE

tier1_count = 0
tier2_count = 0
adaptation_log = []

# Re-initialize drift detectors
ph = PageHinkley(min_instances=60, threshold=5)
ad = ADWIN(delta=0.004)
ks = KSWIN(alpha=0.001, window_size=500, stat_size=150)
methods = {"Page-Hinkley": ph, "ADWIN": ad, "KSWIN": ks}

print("=" * 60)
print("ACF-MAAR Streaming Pipeline - Phase 2")
print("=" * 60)
print(f"Error threshold (δ): {ERROR_THRESHOLD}")
print()

stream_data = reframed.iloc[START_IDX : START_IDX + DAY_SIZE]
target_stream = stream_data['var1(t-90)'].values.flatten()

for j in range(1, DAY_SIZE):
    val = target_stream[j - 1]

    for detector_name, dd in methods.items():
        dd.update(val)

        if dd.drift_detected:
            eval_chunk = reframed.iloc[START_IDX : START_IDX + j]
            eval_X = eval_chunk.drop(columns=['var1(t-90)'])
            eval_y = eval_chunk['var1(t-90)']
            pred = final_best_model.predict(eval_X)
            mae = mean_absolute_error(eval_y, pred)

            if mae > ERROR_THRESHOLD:
                tier2_count += 1
                print(f"[Tier 2] Drift at sample {j} | {detector_name} | "
                      f"MAE: {mae:.4f} > {ERROR_THRESHOLD}")
                print(f"         → Triggering AutoML model replacement...")

                retrain_start = max(0, START_IDX + j - DAY_SIZE)
                retrain_data = reframed.iloc[retrain_start : START_IDX + j]

                final_best_model = automodel(retrain_data)
                model_name = type(final_best_model[-1]).__name__

                adaptation_log.append({
                    'sample': j, 'tier': 2, 'detector': detector_name,
                    'mae': mae, 'model': model_name
                })

                print(f"         → New model: {model_name}")
                print()
            else:
                tier1_count += 1
                print(f"[Tier 1] Drift at sample {j} | {detector_name} | "
                      f"MAE: {mae:.4f} ≤ {ERROR_THRESHOLD} → Continuing")

print()
print("=" * 60)
print("Adaptation Summary")
print("=" * 60)
print(f"Tier 1 responses (window resize only): {tier1_count}")
print(f"Tier 2 responses (model replacement):  {tier2_count}")
print(f"Total drift events handled:            {tier1_count + tier2_count}")

if adaptation_log:
    print(f"\nModel replacements:")
    for entry in adaptation_log:
        print(f"  Sample {entry['sample']}: {entry['detector']} → {entry['model']} (MAE: {entry['mae']:.4f})")

In [ ]:
# --- ACF-MAAR Streaming Pipeline with Asynchronous AutoML ---

DAY_SIZE = 1440
ERROR_THRESHOLD = 0.2
START_IDX = DAY_SIZE

tier1_count = 0
tier2_count = 0
adaptation_log = []
predictions_log = []

# Re-initialize drift detectors
ph = PageHinkley(min_instances=60, threshold=5)
ad = ADWIN(delta=0.004)
ks = KSWIN(alpha=0.001, window_size=500, stat_size=150)
methods = {"Page-Hinkley": ph, "ADWIN": ad, "KSWIN": ks}

# Async model training state
async_model_ready = {"model": None, "is_training": False, "start_time": None}

def async_automl_train(retrain_data, async_state):
    """Run AutoML in a background thread (asynchronous model replacement)."""
    async_state["start_time"] = time.time()
    async_state["is_training"] = True

    new_model = automodel(retrain_data)

    async_state["model"] = new_model
    async_state["is_training"] = False
    elapsed = time.time() - async_state["start_time"]
    model_name = type(new_model[-1]).__name__
    print(f"         ✓ Async AutoML complete in {elapsed:.1f}s → {model_name} ready for deployment")

print("=" * 60)
print("ACF-MAAR Streaming Pipeline - Phase 2 (Async AutoML)")
print("=" * 60)
print(f"Error threshold (δ): {ERROR_THRESHOLD}")
print(f"Async model replacement: ENABLED")
print()

stream_data = reframed.iloc[START_IDX : START_IDX + DAY_SIZE]
target_stream = stream_data['var1(t-90)'].values.flatten()

for j in range(1, DAY_SIZE):
    val = target_stream[j - 1]

    # --- Check if async model is ready to deploy ---
    if async_model_ready["model"] is not None and not async_model_ready["is_training"]:
        old_model_name = type(final_best_model[-1]).__name__
        final_best_model = async_model_ready["model"]
        new_model_name = type(final_best_model[-1]).__name__
        async_model_ready["model"] = None
        print(f"    [Deploy] Sample {j}: Swapped {old_model_name} → {new_model_name} (seamless)")
        print()

    # --- Generate prediction using CURRENT model (never blocked) ---
    pred_row = reframed.iloc[[START_IDX + j - 1]].drop(columns=['var1(t-90)'])
    pred_val = final_best_model.predict(pred_row)[0]
    actual_val = reframed.iloc[START_IDX + j - 1]['var1(t-90)']
    predictions_log.append({'sample': j, 'predicted': pred_val, 'actual': actual_val})

    # --- Drift detection ---
    for detector_name, dd in methods.items():
        dd.update(val)

        if dd.drift_detected:
            eval_chunk = reframed.iloc[START_IDX : START_IDX + j]
            eval_X = eval_chunk.drop(columns=['var1(t-90)'])
            eval_y = eval_chunk['var1(t-90)']
            pred = final_best_model.predict(eval_X)
            mae = mean_absolute_error(eval_y, pred)

            if mae > ERROR_THRESHOLD:
                if not async_model_ready["is_training"]:
                    tier2_count += 1
                    print(f"[Tier 2] Drift at sample {j} | {detector_name} | "
                          f"MAE: {mae:.4f} > {ERROR_THRESHOLD}")
                    print(f"         → Launching async AutoML (predictions continue with current model)...")

                    retrain_start = max(0, START_IDX + j - DAY_SIZE)
                    retrain_data = reframed.iloc[retrain_start : START_IDX + j].copy()

                    train_thread = threading.Thread(
                        target=async_automl_train,
                        args=(retrain_data, async_model_ready)
                    )
                    train_thread.start()

                    adaptation_log.append({
                        'sample': j, 'tier': 2, 'detector': detector_name,
                        'mae': mae, 'async': True
                    })
                else:
                    print(f"[Tier 2] Drift at sample {j} | {detector_name} | "
                          f"MAE: {mae:.4f} → AutoML already running, skipping")
            else:
                tier1_count += 1
                print(f"[Tier 1] Drift at sample {j} | {detector_name} | "
                      f"MAE: {mae:.4f} ≤ {ERROR_THRESHOLD} → Continuing")

# Wait for any remaining async training
if async_model_ready["is_training"]:
    print("\nWaiting for final async AutoML to complete...")
    while async_model_ready["is_training"]:
        time.sleep(0.5)
    final_best_model = async_model_ready["model"]
    print("Final model deployed.")

# --- Summary ---
print()
print("=" * 60)
print("Adaptation Summary")
print("=" * 60)
print(f"Tier 1 responses (window resize only):  {tier1_count}")
print(f"Tier 2 responses (async model replace):  {tier2_count}")
print(f"Total drift events handled:              {tier1_count + tier2_count}")
print(f"Total predictions generated:             {len(predictions_log)}")

pred_df = pd.DataFrame(predictions_log)
stream_mae = mean_absolute_error(pred_df['actual'], pred_df['predicted'])
stream_r2 = r2_score(pred_df['actual'], pred_df['predicted'])
print(f"\nStreaming Prediction Performance:")
print(f"  MAE:  {stream_mae:.4f}")
print(f"  R²:   {stream_r2:.4f}")

if adaptation_log:
    print(f"\nModel replacements (all async):")
    for entry in adaptation_log:
        print(f"  Sample {entry['sample']}: {entry['detector']} triggered (MAE: {entry['mae']:.4f})")

---
## 9. Baseline: Online Linear Regression (River)

Online linear regression using the River library as a baseline comparison.  
The model updates incrementally with each observation and uses ADWIN for drift-triggered resets.

**Reference:** Section 4.5.1 (Comparison with Conventional Regression Models)

In [ ]:
# --- Online Linear Regression baseline ---
model_lr = LinearRegression(optimizer=SGD())
metric_lr = MSE()
drift_detector_lr = drift.ADWIN()

lr_df = reframed.iloc[0:1440]
X_lr = lr_df.drop(columns=['var1(t-90)'])
y_lr = lr_df['var1(t-90)']

data_stream = iter_pandas(X_lr, y_lr)
lr_mse_history = []
lr_drift_points = []

print("Running Online Linear Regression baseline...")
print()

for i, (x_obs, y_true) in enumerate(data_stream):
    y_pred = model_lr.predict_one(x_obs)

    if y_pred is not None:
        metric_lr.update(y_true, y_pred)
        lr_mse_history.append(metric_lr.get())

        error = abs(y_true - y_pred)
        drift_detector_lr.update(error)
        if drift_detector_lr.drift_detected:
            lr_drift_points.append(i + 1)
            print(f"  Drift detected at sample {i + 1}, resetting model")
            model_lr = LinearRegression(optimizer=SGD())

    model_lr.learn_one(x_obs, y_true)

    if (i + 1) % 360 == 0:
        print(f"  Samples: {i + 1:>5}, MSE: {metric_lr.get():.6f}")

print()
print(f"Final MSE: {metric_lr.get():.6f}")
print(f"Total drifts detected: {len(lr_drift_points)}")

---
## 10. Stream-Specific Baselines (HTR, ARF, OBR, KNN)

Compare ACF-MAAR against stream-specific regression models from the River library.

**Reference:** Section 4.5.2 (Comparison with Stream-Specific Regression Models)

In [ ]:
from river.tree import HoeffdingTreeRegressor
from river.forest import ARFRegressor
from river.neighbors import KNNRegressor
from river.linear_model import BayesianLinearRegression
from river.metrics import MAE as RiverMAE, R2

stream_models = {
    "Hoeffding Tree (HTR)": HoeffdingTreeRegressor(),
    "Adaptive Random Forest (ARF)": ARFRegressor(n_models=10, seed=42),
    "Online Bayesian Regression (OBR)": BayesianLinearRegression(),
    "KNN Regressor": KNNRegressor(n_neighbors=5),
}

eval_data = reframed.iloc[0:1440]
X_eval = eval_data.drop(columns=['var1(t-90)'])
y_eval_stream = eval_data['var1(t-90)']

print("Stream-Specific Model Comparison:")
print(f"{'Model':<35} {'MAE':>8} {'R²':>8} {'Time(ms)':>10}")
print("-" * 65)

for model_name, model in stream_models.items():
    mae_metric = RiverMAE()
    r2_metric = R2()

    start_time = time.time()

    data_stream = iter_pandas(X_eval, y_eval_stream)
    for x_obs, y_true in data_stream:
        y_pred = model.predict_one(x_obs)
        if y_pred is not None:
            mae_metric.update(y_true, y_pred)
            r2_metric.update(y_true, y_pred)
        model.learn_one(x_obs, y_true)

    elapsed = (time.time() - start_time) * 1000

    print(f"{model_name:<35} {mae_metric.get():>8.4f} {r2_metric.get():>8.4f} {elapsed:>10.2f}")

print()
print("ACF-MAAR results from streaming pipeline above for comparison.")

---
## 11. ACF-Driven Adaptive Window Sizing

Demonstrate how the ACF profile determines different window sizes for different temporal regimes.

**Reference:** Section 3.2 (ACF-Driven Adaptive Window Sizing) & Section 4.3

In [ ]:
def acf_window_size(data, alpha=0.05, max_lags=150, consecutive_c=3):
    """
    Determine adaptive window size from ACF profile.
    Window = first lag where c consecutive ACF values fall below Bartlett bound.

    Reference: Eq. (7) in the paper.
    """
    n = len(data)
    bartlett_bound = 1.96 / np.sqrt(n)

    acf_values = acf(data, nlags=min(max_lags, n // 4), fft=True)

    count = 0
    for k in range(1, len(acf_values)):
        if abs(acf_values[k]) < bartlett_bound:
            count += 1
            if count >= consecutive_c:
                return k - consecutive_c + 1, acf_values, bartlett_bound
        else:
            count = 0

    return len(acf_values), acf_values, bartlett_bound

# --- Stable regime: weekday ---
stable_data = df_minute['Global_active_power'].iloc[0:1440].dropna().values
n_stable, acf_stable, bound_stable = acf_window_size(stable_data)

# --- Volatile regime: weekend ---
volatile_data = df_minute['Global_active_power'].iloc[7200:8640].dropna().values
n_volatile, acf_volatile, bound_volatile = acf_window_size(volatile_data)

print(f"Stable regime:   n* = {n_stable} minutes")
print(f"Volatile regime: n* = {n_volatile} minutes")
print(f"Bartlett bound (stable):   ±{bound_stable:.4f}")
print(f"Bartlett bound (volatile): ±{bound_volatile:.4f}")

In [ ]:
# --- Plot ACF profiles for both regimes (Fig. 6 in paper) ---
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(stable_data, color='steelblue', linewidth=0.5)
axes[0, 0].axvline(x=n_stable, color='red', linestyle='--', label=f'n* = {n_stable} min')
axes[0, 0].set_title('(a) Stable Regime — Time Series')
axes[0, 0].set_xlabel('Time (minutes)')
axes[0, 0].set_ylabel('Global Active Power (kW)')
axes[0, 0].legend()

axes[0, 1].bar(range(len(acf_stable)), acf_stable, color='steelblue', width=1.0, alpha=0.7)
axes[0, 1].axhline(y=bound_stable, color='gray', linestyle='--', label=f'Bartlett bound (±{bound_stable:.3f})')
axes[0, 1].axhline(y=-bound_stable, color='gray', linestyle='--')
axes[0, 1].axvline(x=n_stable, color='red', linestyle='--', label=f'n* = {n_stable}')
axes[0, 1].set_title('(b) ACF Profile — Stable Regime')
axes[0, 1].set_xlabel('Lag k')
axes[0, 1].set_ylabel('Autocorrelation')
axes[0, 1].legend(fontsize=8)

axes[1, 0].plot(volatile_data, color='darkorange', linewidth=0.5)
axes[1, 0].axvline(x=n_volatile, color='red', linestyle='--', label=f'n* = {n_volatile} min')
axes[1, 0].set_title('(c) Volatile Regime — Time Series')
axes[1, 0].set_xlabel('Time (minutes)')
axes[1, 0].set_ylabel('Global Active Power (kW)')
axes[1, 0].legend()

axes[1, 1].bar(range(len(acf_volatile)), acf_volatile, color='darkorange', width=1.0, alpha=0.7)
axes[1, 1].axhline(y=bound_volatile, color='gray', linestyle='--', label=f'Bartlett bound (±{bound_volatile:.3f})')
axes[1, 1].axhline(y=-bound_volatile, color='gray', linestyle='--')
axes[1, 1].axvline(x=n_volatile, color='red', linestyle='--', label=f'n* = {n_volatile}')
axes[1, 1].set_title('(d) ACF Profile — Volatile Regime')
axes[1, 1].set_xlabel('Lag k')
axes[1, 1].set_ylabel('Autocorrelation')
axes[1, 1].legend(fontsize=8)

plt.suptitle('ACF-Driven Adaptive Window Sizing Across Two Temporal Regimes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 12. ACF-Based Structural Drift Detection

Compute ACF distance between consecutive windows to detect structural drifts.

**Reference:** Section 3.3 (Eq. 8, 9, 10)

In [ ]:
def acf_distance(window1, window2, max_lags=60):
    """
    Compute Euclidean distance between ACF profiles of two windows.
    Reference: Eq. (8) in the paper.
    """
    acf1 = acf(window1, nlags=max_lags, fft=True)[1:]
    acf2 = acf(window2, nlags=max_lags, fft=True)[1:]
    return np.sqrt(np.sum((acf1 - acf2) ** 2)), acf1, acf2

def acf_drift_threshold(n, L, alpha=0.05):
    """
    Compute statistically derived drift threshold.
    Reference: Eq. (10) in the paper.
    """
    chi2_val = chi2.ppf(1 - alpha, df=L)
    return np.sqrt((2 * L / n) * chi2_val)

# --- Detect structural drifts across sliding windows ---
target_series = df_minute['Global_active_power'].dropna().values
WINDOW_SIZE = 1440
MAX_LAGS = 60
ALPHA = 0.05
NUM_WINDOWS = 10

print("ACF-Based Structural Drift Detection:")
print(f"Window size: {WINDOW_SIZE}, Max lags: {MAX_LAGS}, Alpha: {ALPHA}")
print()
print(f"{'Window Pair':<20} {'ACF Distance':>12} {'Threshold':>12} {'Drift?':>8}")
print("-" * 55)

drift_points = []

for i in range(NUM_WINDOWS - 1):
    w1_start = i * WINDOW_SIZE
    w1_end = w1_start + WINDOW_SIZE
    w2_start = w1_end
    w2_end = w2_start + WINDOW_SIZE

    if w2_end > len(target_series):
        break

    w1 = target_series[w1_start:w1_end]
    w2 = target_series[w2_start:w2_end]

    d_acf, _, _ = acf_distance(w1, w2, MAX_LAGS)
    tau = acf_drift_threshold(WINDOW_SIZE, MAX_LAGS, ALPHA)

    is_drift = d_acf > tau
    if is_drift:
        drift_points.append(i + 1)

    print(f"W{i+1} vs W{i+2}          {d_acf:>12.4f} {tau:>12.4f} {'YES ⚠' if is_drift else 'NO':>8}")

print()
print(f"Total structural drifts detected: {len(drift_points)}")

---
## 13. Drift Visualization

Plot the data stream with detected drift points highlighted.

**Reference:** Figure 7 in the paper

In [ ]:
# --- Drift point visualization (Fig. 7) ---
plot_length = NUM_WINDOWS * WINDOW_SIZE
plot_data = target_series[:plot_length]

plt.figure(figsize=(14, 4))
plt.plot(plot_data, color='steelblue', linewidth=0.3, label='Data stream')

for dp in drift_points:
    drift_idx = dp * WINDOW_SIZE
    plt.axvline(x=drift_idx, color='red', linestyle='--', alpha=0.5)
    plt.scatter(drift_idx, plot_data[drift_idx], color='red', s=50, zorder=5)

plt.scatter([], [], color='red', s=50, label='Drift detected')
plt.xlabel('Time (minutes)')
plt.ylabel('Global Active Power (kW)')
plt.title('Concept Drift Visualization in the Household Data Stream')
plt.legend()
plt.tight_layout()
plt.show()

---
## 14. Forecasting Performance Visualization

Plot ground truth vs. predicted values to demonstrate tracking quality.

**Reference:** Figure 8 in the paper

In [ ]:
# --- Ground truth vs predicted (Fig. 8) ---
pred_df = pd.DataFrame(predictions_log)

plt.figure(figsize=(14, 4))
plt.plot(pred_df['actual'].values, color='steelblue', linewidth=0.8, label='Ground Truth', alpha=0.8)
plt.plot(pred_df['predicted'].values, color='darkorange', linewidth=0.8, label='Predicted', alpha=0.8)

for entry in adaptation_log:
    plt.axvline(x=entry['sample'], color='red', linestyle='--', alpha=0.4)

plt.xlabel('Sample')
plt.ylabel('Global Active Power (kW)')
plt.title('Forecasting Performance: Ground Truth vs. Predicted Values')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Streaming MAE:  {mean_absolute_error(pred_df['actual'], pred_df['predicted']):.4f}")
print(f"Streaming R²:   {r2_score(pred_df['actual'], pred_df['predicted']):.4f}")
print(f"Streaming MAPE: {mean_absolute_percentage_error(pred_df['actual'], pred_df['predicted']):.4f}")

---
## 15. Fixed Window Comparison

Compare ACF-adaptive windowing against fixed window sizes (30, 45, 90, 120).

**Reference:** Section 4.3.2 & Table 4

In [ ]:
# --- Fixed window comparison (Table 4) ---
fixed_windows = [30, 45, 90, 120]
results = []

print("Evaluating fixed window sizes vs ACF-adaptive...")
print(f"{'Window':<15} {'MAE':>8} {'MSE':>8} {'R²':>8} {'MAPE':>8}")
print("-" * 52)

for ws in fixed_windows:
    # Re-create supervised frame with THIS window's lag count
    reframed_ws = series_to_supervised(newdf.values, n_in=ws, n_out=1)
    target_col = f'var1(t-{ws})'

    train_data = reframed_ws.iloc[0:1440]

    s = setup(data=train_data, target=target_col, session_id=123, verbose=False)
    models_to_compare = ['et', 'xgboost', 'rf', 'gbr', 'huber']
    best = compare_models(include=models_to_compare, sort='MAE', fold=5, verbose=False)
    model = finalize_model(best)

    # Evaluate on day 2
    eval_data = reframed_ws.iloc[1440:2880]
    eval_X = eval_data.drop(columns=[target_col])
    eval_y = eval_data[target_col]
    preds = model.predict(eval_X)

    mae = mean_absolute_error(eval_y, preds)
    mse = mean_squared_error(eval_y, preds)
    r2 = r2_score(eval_y, preds)
    mape = mean_absolute_percentage_error(eval_y, preds)

    results.append({'window': f'Fixed-{ws}', 'mae': mae, 'mse': mse, 'r2': r2, 'mape': mape})
    print(f"Fixed-{ws:<9} {mae:>8.4f} {mse:>8.4f} {r2:>8.4f} {mape:>8.4f}")

# --- ACF-adaptive window ---
stable_data = newdf['Global_active_power'].iloc[0:1440].dropna().values
acf_lags, _, _ = acf_window_size(stable_data)

reframed_acf = series_to_supervised(newdf.values, n_in=acf_lags, n_out=1)
target_col_acf = f'var1(t-{acf_lags})'

train_acf = reframed_acf.iloc[0:1440]
s = setup(data=train_acf, target=target_col_acf, session_id=123, verbose=False)
models_to_compare = ['et', 'xgboost', 'rf', 'gbr', 'huber']
best = compare_models(include=models_to_compare, sort='MAE', fold=5, verbose=False)
model_acf = finalize_model(best)

eval_acf = reframed_acf.iloc[1440:2880]
eval_X_acf = eval_acf.drop(columns=[target_col_acf])
eval_y_acf = eval_acf[target_col_acf]
preds_acf = model_acf.predict(eval_X_acf)

mae_acf = mean_absolute_error(eval_y_acf, preds_acf)
mse_acf = mean_squared_error(eval_y_acf, preds_acf)
r2_acf = r2_score(eval_y_acf, preds_acf)
mape_acf = mean_absolute_percentage_error(eval_y_acf, preds_acf)

results.append({'window': f'ACF-window(n*={acf_lags})', 'mae': mae_acf, 'mse': mse_acf, 'r2': r2_acf, 'mape': mape_acf})
print(f"ACF-win(n*={acf_lags}){'':<4} {mae_acf:>8.4f} {mse_acf:>8.4f} {r2_acf:>8.4f} {mape_acf:>8.4f}")

# --- ACF-MAAR full streaming pipeline ---
stream_mae = mean_absolute_error(pred_df['actual'], pred_df['predicted'])
stream_mse = mean_squared_error(pred_df['actual'], pred_df['predicted'])
stream_r2 = r2_score(pred_df['actual'], pred_df['predicted'])
stream_mape = mean_absolute_percentage_error(pred_df['actual'], pred_df['predicted'])

results.append({'window': 'ACF-MAAR', 'mae': stream_mae, 'mse': stream_mse, 'r2': stream_r2, 'mape': stream_mape})
print(f"{'ACF-MAAR':<15} {stream_mae:>8.4f} {stream_mse:>8.4f} {stream_r2:>8.4f} {stream_mape:>8.4f}")

results_df = pd.DataFrame(results)
print("\n")
print(results_df.to_string(index=False))

---
## 16. Ablation Study

Remove each ACF component one at a time to measure its contribution.

**Reference:** Section 4.7 & Table 12

In [ ]:
# --- Ablation Study ---
from copy import deepcopy

DAY_SIZE = 1440
ERROR_THRESHOLD = 0.2
START_IDX = DAY_SIZE

def run_ablation_variant(reframed, newdf, variant_name, use_acf_window=True,
                          use_acf_drift=True, use_acf_trigger=True):
    """
    Run one ablation variant of ACF-MAAR.

    Parameters
    ----------
    use_acf_window : bool - If False, use fixed window=60 instead of ACF-derived window
    use_acf_drift  : bool - If False, use only ADWIN instead of all 3 detectors
    use_acf_trigger: bool - If False, use fixed 10% error threshold instead of adaptive
    """

    # --- Window sizing ---
    if use_acf_window:
        stable_data = newdf['Global_active_power'].iloc[0:1440].dropna().values
        n_lags, _, _ = acf_window_size(stable_data)
    else:
        n_lags = 60

    reframed_abl = series_to_supervised(newdf.values, n_in=n_lags, n_out=1)
    target_col = f'var1(t-{n_lags})'

    # --- Train initial model ---
    train_data = reframed_abl.iloc[0:DAY_SIZE]
    s = setup(data=train_data, target=target_col, session_id=123, verbose=False)
    models_to_compare = ['et', 'xgboost', 'rf', 'gbr', 'huber']
    best = compare_models(include=models_to_compare, sort='MAE', fold=5, verbose=False)
    current_model = finalize_model(best)

    # --- Drift detectors ---
    if use_acf_drift:
        ph = PageHinkley(min_instances=60, threshold=5)
        ad = ADWIN(delta=0.004)
        ks = KSWIN(alpha=0.001, window_size=500, stat_size=150)
        detectors = {"Page-Hinkley": ph, "ADWIN": ad, "KSWIN": ks}
    else:
        ad = ADWIN(delta=0.004)
        detectors = {"ADWIN": ad}

    # --- Error threshold ---
    if use_acf_trigger:
        threshold = ERROR_THRESHOLD
    else:
        threshold = 0.10

    # --- Stream through Day 2 ---
    tier2_abl = 0
    predictions_abl = []

    stream_data = reframed_abl.iloc[START_IDX : START_IDX + DAY_SIZE]

    if len(stream_data) == 0:
        print(f"  WARNING: No streaming data available for {variant_name}")
        return None, None, 0

    target_stream = stream_data[target_col].values.flatten()

    for j in range(1, min(DAY_SIZE, len(stream_data))):
        val = target_stream[j - 1]

        row = stream_data.iloc[j:j+1]
        row_X = row.drop(columns=[target_col])
        row_y = row[target_col].values[0]
        pred = current_model.predict(row_X)[0]
        predictions_abl.append({'actual': row_y, 'predicted': pred})

        for det_name, dd in detectors.items():
            dd.update(val)
            if dd.drift_detected:
                eval_chunk = reframed_abl.iloc[START_IDX : START_IDX + j]
                eval_X_chunk = eval_chunk.drop(columns=[target_col])
                eval_y_chunk = eval_chunk[target_col]
                chunk_pred = current_model.predict(eval_X_chunk)
                mae_check = mean_absolute_error(eval_y_chunk, chunk_pred)

                if mae_check > threshold:
                    tier2_abl += 1
                    retrain_start = max(0, START_IDX + j - DAY_SIZE)
                    retrain_data = reframed_abl.iloc[retrain_start : START_IDX + j]

                    s = setup(data=retrain_data, target=target_col,
                              session_id=123, verbose=False)
                    best = compare_models(include=models_to_compare, sort='MAE',
                                          fold=5, verbose=False)
                    current_model = finalize_model(best)

    pred_abl_df = pd.DataFrame(predictions_abl)
    abl_mae = mean_absolute_error(pred_abl_df['actual'], pred_abl_df['predicted'])
    abl_r2 = r2_score(pred_abl_df['actual'], pred_abl_df['predicted'])

    return abl_mae, abl_r2, tier2_abl


# === Run Ablation Study ===
print("=" * 68)
print("Ablation Study (Table 5)")
print("=" * 68)
print(f"{'Configuration':<35} {'MAE':>8} {'R²':>8} {'Adaptations':>12}")
print("-" * 68)

print(f"{'Full ACF-MAAR':<35} {stream_mae:>8.4f} {stream_r2:>8.4f} {tier2_count:>12}")

print("\nRunning: Without ACF window (fixed=60)...")
mae_no_win, r2_no_win, t2_no_win = run_ablation_variant(
    reframed, newdf, "No ACF window",
    use_acf_window=False, use_acf_drift=True, use_acf_trigger=True
)
print(f"{'- ACF window (fixed 60)':<35} {mae_no_win:>8.4f} {r2_no_win:>8.4f} {t2_no_win:>12}")

print("\nRunning: Without ACF drift (ADWIN only)...")
mae_no_drift, r2_no_drift, t2_no_drift = run_ablation_variant(
    reframed, newdf, "No ACF drift",
    use_acf_window=True, use_acf_drift=False, use_acf_trigger=True
)
print(f"{'- ACF drift (ADWIN only)':<35} {mae_no_drift:>8.4f} {r2_no_drift:>8.4f} {t2_no_drift:>12}")

print("\nRunning: Without ACF trigger (fixed 10%)...")
mae_no_trig, r2_no_trig, t2_no_trig = run_ablation_variant(
    reframed, newdf, "No ACF trigger",
    use_acf_window=True, use_acf_drift=True, use_acf_trigger=False
)
print(f"{'- ACF trigger (fixed 10%)':<35} {mae_no_trig:>8.4f} {r2_no_trig:>8.4f} {t2_no_trig:>12}")

print("\nRunning: Without all ACF (Fixed-MAAR)...")
mae_no_all, r2_no_all, t2_no_all = run_ablation_variant(
    reframed, newdf, "Fixed-MAAR",
    use_acf_window=False, use_acf_drift=False, use_acf_trigger=False
)
print(f"{'- All ACF (Fixed-MAAR)':<35} {mae_no_all:>8.4f} {r2_no_all:>8.4f} {t2_no_all:>12}")

# === Summary Table ===
print("\n" + "=" * 68)
print("Ablation Summary")
print("=" * 68)
print(f"{'Configuration':<35} {'MAE':>8} {'R²':>8} {'Adaptations':>12}")
print("-" * 68)
ablation_results = [
    ("Full ACF-MAAR",              stream_mae,    stream_r2,    tier2_count),
    ("- ACF window (fixed 60)",    mae_no_win,    r2_no_win,    t2_no_win),
    ("- ACF drift (ADWIN only)",   mae_no_drift,  r2_no_drift,  t2_no_drift),
    ("- ACF trigger (fixed 10%)",  mae_no_trig,   r2_no_trig,   t2_no_trig),
    ("- All ACF (Fixed-MAAR)",     mae_no_all,    r2_no_all,    t2_no_all),
]

for name, mae_val, r2_val, adapt in ablation_results:
    print(f"{name:<35} {mae_val:>8.4f} {r2_val:>8.4f} {adapt:>12}")

print("-" * 68)
print("\nEach row removes one ACF component to measure its contribution.")
print("Full ACF-MAAR should outperform all ablated variants.")